In [ ]:
# 데이터 생성
import pandas as pd

dataset = pd.read_csv('../../data/csv/movie_reviews.csv')
dataset.head()

,label,review
0,1,진짜 내 인생 영화입니다. 영상미랑 음악이 미쳤어요.
1,1,배우들 연기력이 장난 아니네요. 몰입감 최고였습니다.
2,0,시간 아까워요. 돈 내고 보기에는 너무 아깝습니다.
3,0,개연성이 하나도 없어서 보는 내내 어이가 없었네요.
4,1,가족들이랑 보기 딱 좋은 따뜻한 영화입니다. 강추!


In [37]:
# 독립 종속(답) 분리

X = dataset['review'].to_numpy()
y = dataset['label'].to_numpy().astype('float32')


In [47]:
# 텍스트 전처리
# 형태소 분석

from konlpy.tag import Okt
okt = Okt()

# ---------------------------------------------------------------
# 문장을 형태소 별로 분리해서 공백으로 구분하여 다시 한문장으로 만드는 함수
# ---------------------------------------------------------------

# def tokenize_korean(text_list):
# 	result = []
# 	for text in text_list:
# 		# morphs = okt.morphs(text, stem=True)
# 		new_text = " ".join(okt.morphs(text, stem=True))
# 		result.append(new_text)
# 	return result

def tokenize_korean(text_list):
 return [" ".join(okt.morphs(text, stem=True)) for text in text_list]
# print(tokenize_korean(["오늘은 학교에 갔습니다."]))
korean_morphs = tokenize_korean(X)

In [49]:
# 벡터화
from tensorflow.keras import layers
vectorize_layer = layers.TextVectorization(
  max_tokens=1000,
  output_mode = 'int',
  output_sequence_length=10
)

# 데이터를 학습 시켜 단어 사전을 만듬
vectorize_layer.adapt(korean_morphs)

In [40]:
# 파이프라인 적용(웬만하면 하는게 좋음)
# X를 학습시킬 때 만주열 이슈로 인해 에러가 발생
# 파이프라인을 적용하면 텐서플로우의 텐서 객체로 변환되기 때문에 이슈가 해결

import tensorflow as tf

# from_tensor_slices : korean_morphs와 y에서  하나씩 꺼내 하나의 세트로 묶음
train_ds = tf.data.Dataset.from_tensor_slices((korean_morphs, y)).batch(8)

In [42]:
# 어텐션 모델 설계 -> models.Sequential()을 사용 못함 -> 입력1 -> 출력1인 경우만 가능
# 그런데 어텐션에서 중간에 입력을 3개로 햇다가 하나로 합쳐야함 -> 함수형 API를 이용

from tensorflow.keras import models

# 함수형 API의 특징. 출력을 직접 입력으로 넣어줘야 함
def build_attention_model():
  # 인풋 층 설정
  inputs = layers.Input(shape=(1,), dtype=tf.string)
  
	# 단어 사전 (숫자로만 됨)
  x = vectorize_layer(inputs)
  
	# 단어장 크기
  vacab_size = vectorize_layer.vocabulary_size()
  
	# 벡터화
  embed = layers.Embedding(input_dim=vacab_size, output_dim=64)(x)
  
	# 어텐션
  query = layers.Dense(64)(embed)
  value = layers.Dense(64)(embed)
  
	# key와 value를 같게 하려면 하나를 추가하지 않으면 됨
  attention_output = layers.Attention()([query, value])
  
	# 요약
  x = layers.GlobalAveragePooling1D()(attention_output)
  
	# 출력층
  # sigmoid인 이유 : 예제가 리뷰의 긍정 부정을 판별하기 때문에
  outputs = layers.Dense(1, activation='sigmoid')(x)	
  
  return models.Model(inputs=inputs, outputs=outputs)

# 모델 설계
model = build_attention_model()
  

In [50]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [51]:
model.fit(train_ds, epochs=50, verbose=0)

In [52]:
# 테스트

test_text = ["액션이 화려해요", "액션이 화려해서 재밌었어요", "액션이 화려하지만 지루해요"]

# 형태소별로 문장을 수정
test_morphs = tokenize_korean(test_text)
# test_morphs

# 테스트 할 때 텐서 타입을 변환(문자열 이슈)
test_input = tf.constant(test_morphs)

# 예측 실행
predictions = model.predict(test_input)
predictions_size = len(predictions)
for i in range(predictions_size):
  p = predictions[i][0] * 100
  print(f'{test_text[i]} : {'긍정' if p > 50 else '부정'}({p:.2f}%)')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step
액션이 화려해요 : 긍정(79.15%)
액션이 화려해서 재밌었어요 : 긍정(73.85%)
액션이 화려하지만 지루해요 : 부정(42.07%)
